# 3단계 V5-2. 보이스피싱 단독 EDA·머신러닝 분석

정상 금융상담을 사용하지 않고 금융감독원 보이스피싱 사례 내부의 차이와 유사성을 분석합니다.

> `구축 데이터셋_v4` → 보이스피싱 품질·EDA → 사칭·행동·심리·금액 분석 → 대출사기형/수사기관형 분류 → 행동·심리 전술 군집 → 텍스트 유사 사건 군집 → 보고서 저장

## 분석 질문

1. 대출사기형과 수사기관사칭형은 사칭 대상·요구 행동·심리전략이 다른가?
2. 보이스피싱 텍스트만으로 두 유형을 어느 정도 구분할 수 있는가?
3. 기존 폴더 분류와 별개로 유사한 전술·문구를 가진 하위 군집이 존재하는가?
4. 피해자에게 요구한 금액과 피해자에게 제시한 금액은 어떤 용도와 분포를 보이는가?

사칭·행동·심리·금액은 자동 추출한 `SILVER` 라벨입니다. 실제 피해 여부 정답이 없으므로 피해 발생 확률을 예측하지 않습니다.


In [ ]:
# 0. 라이브러리 설치
!pip -q install pandas pyarrow scikit-learn scipy seaborn matplotlib koreanize-matplotlib joblib openpyxl


In [ ]:
# 1. 라이브러리와 Google Drive
from google.colab import drive
from pathlib import Path
from IPython.display import display
import hashlib, json, re, unicodedata, warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
from scipy.stats import chi2_contingency
from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, adjusted_rand_score, classification_report,
    confusion_matrix, precision_recall_fscore_support, silhouette_score)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
drive.mount('/content/drive')
print('Google Drive 연결 완료')


## 1. 경로와 설정값


In [ ]:
# 2. 구축 데이터셋_v4와 보이스피싱 단독 결과 폴더
DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_ROOT = DRIVE_ROOT / '보이스피싱_분석'
DATASET_ROOT = PROJECT_ROOT / '구축 데이터셋_v4'
STANDARD_ROOT = DATASET_ROOT / '01_standard_tables'
ML_ROOT = DATASET_ROOT / '02_ml_tables'
OUTPUT_ROOT = PROJECT_ROOT / '머신러닝_분석결과_v5_2_보이스피싱단독'
TABLE_ROOT = OUTPUT_ROOT / '01_분석표'
FIGURE_ROOT = OUTPUT_ROOT / '02_그래프'
SPLIT_ROOT = OUTPUT_ROOT / '03_데이터분리'
MODEL_ROOT = OUTPUT_ROOT / '04_모델'
PRED_ROOT = OUTPUT_ROOT / '05_예측군집결과'
REPORT_ROOT = OUTPUT_ROOT / '06_보고서'
for folder in [TABLE_ROOT, FIGURE_ROOT, SPLIT_ROOT, MODEL_ROOT, PRED_ROOT, REPORT_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

SEED = 42
TEST_RATIO = 0.20
DEV_RATIO = 0.20
MIN_TEXT_LENGTH = 10
MAX_FEATURES = 50000
TARGETS = ['LOAN_FRAUD', 'INSTITUTION_IMPERSONATION']
TARGET_KO = {'LOAN_FRAUD':'대출사기형', 'INSTITUTION_IMPERSONATION':'수사기관 사칭형'}
assert STANDARD_ROOT.exists() and ML_ROOT.exists(), f'구축 데이터셋_v4 경로를 확인하세요: {DATASET_ROOT}'
print('입력:', DATASET_ROOT)
print('출력:', OUTPUT_ROOT)


## 2. 보이스피싱 데이터만 불러오기


In [ ]:
# 3. 정상상담 테이블은 읽지 않습니다.
def read_table(folder, name, required=True):
    pq = folder / f'{name}.parquet'
    csv = folder / f'{name}.csv'
    if pq.exists(): return pd.read_parquet(pq)
    if csv.exists(): return pd.read_csv(csv, encoding='utf-8-sig')
    if required: raise FileNotFoundError(f'{name}을 찾지 못했습니다: {folder}')
    return pd.DataFrame()

tables = {
    'cases': read_table(STANDARD_ROOT, 'vp_cases'),
    'utterances': read_table(STANDARD_ROOT, 'vp_utterances'),
    'impersonations': read_table(STANDARD_ROOT, 'vp_impersonations'),
    'actions': read_table(STANDARD_ROOT, 'vp_requested_actions'),
    'strategies': read_table(STANDARD_ROOT, 'vp_strategy_events'),
    'amounts': read_table(STANDARD_ROOT, 'vp_amount_events'),
    'fraud_type_ml': read_table(ML_ROOT, 'fraud_type_ml'),
    'clustering_ml': read_table(ML_ROOT, 'case_clustering_ml'),
}
inventory = pd.DataFrame([
    {'테이블':name, '행수':len(df), '컬럼수':len(df.columns)} for name, df in tables.items()
])
display(inventory)
inventory.to_csv(TABLE_ROOT/'00_보이스피싱단독_테이블목록.csv', index=False, encoding='utf-8-sig')
assert len(tables['cases']) > 0 and len(tables['fraud_type_ml']) > 0
print('정상 금융상담 데이터 사용: 0건')


## 3. 데이터 품질과 기본 분포


In [ ]:
# 4. 사건·발화·라벨 품질 확인
cases = tables['cases'].copy()
type_df = tables['fraud_type_ml'].copy()
type_map = type_df[['case_id','supervised_target']].drop_duplicates('case_id')
quality_rows = []
for name, df in tables.items():
    quality_rows.append({
        '테이블':name, '행수':len(df), '완전중복행':int(df.duplicated().sum()),
        '전체결측셀':int(df.isna().sum().sum())
    })
quality_df = pd.DataFrame(quality_rows)
display(quality_df)
quality_df.to_csv(TABLE_ROOT/'01_데이터품질_요약.csv', index=False, encoding='utf-8-sig')

type_count = type_df['supervised_target'].value_counts().rename_axis('사기유형').reset_index(name='사건수')
type_count['한글유형'] = type_count['사기유형'].map(TARGET_KO)
display(type_count)
plt.figure(figsize=(7,5))
sns.barplot(data=type_count, x='한글유형', y='사건수', color='#4c72b0')
plt.title('보이스피싱 유형별 사건 수'); plt.xlabel(''); plt.tight_layout()
plt.savefig(FIGURE_ROOT/'01_보이스피싱_유형분포.png', dpi=170); plt.show()


In [ ]:
# 5. 사건 길이·발화 수·범인 추정 발화 비율
numeric_columns = [c for c in ['duration_sec','turn_count','speaker_count','offender_turn_count','victim_turn_count'] if c in cases.columns]
case_desc = cases.merge(type_map, on='case_id', how='left', suffixes=('','_type'))
case_desc = case_desc[case_desc['supervised_target'].isin(TARGETS)].copy()
if {'offender_turn_count','turn_count'}.issubset(case_desc.columns):
    case_desc['offender_turn_ratio'] = case_desc['offender_turn_count'].div(case_desc['turn_count'].replace(0,np.nan))
    numeric_columns.append('offender_turn_ratio')
desc = case_desc.groupby('supervised_target')[numeric_columns].agg(['count','mean','median','std']).round(3)
display(desc)
desc.to_csv(TABLE_ROOT/'02_유형별_사건발화_기술통계.csv', encoding='utf-8-sig')
print('주의: 범인 발화 비율은 대화 주도권의 확정값이 아니라 발화 배분의 보조 지표입니다.')


## 4. 사건 단위 사칭기관 분석


In [ ]:
# 6. 사건 단위 주요 사칭기관과 판정등급
required_imp = {'primary_impersonation_subtype','primary_impersonation_confidence_tier','impersonation_review_required'}
assert required_imp.issubset(case_desc.columns), f'v4 사칭 컬럼 누락: {required_imp-set(case_desc.columns)}'
case_desc['사기유형'] = case_desc['supervised_target'].map(TARGET_KO)
case_desc['판정등급'] = case_desc['primary_impersonation_confidence_tier'].fillna('UNKNOWN')
confidence_count = pd.crosstab(case_desc['사기유형'], case_desc['판정등급'], margins=True)
confidence_ratio = pd.crosstab(case_desc['사기유형'], case_desc['판정등급'], normalize='index').mul(100).round(1)
display(confidence_count); display(confidence_ratio)
confidence_count.to_csv(TABLE_ROOT/'03_사칭기관_판정등급_건수.csv', encoding='utf-8-sig')
confidence_ratio.to_csv(TABLE_ROOT/'04_사칭기관_판정등급_비율.csv', encoding='utf-8-sig')

reliable_imp = case_desc[case_desc['판정등급'].isin(['HIGH','MEDIUM'])].copy()
imp_ratio = pd.crosstab(reliable_imp['사기유형'], reliable_imp['primary_impersonation_subtype'], normalize='index').mul(100).round(1)
display(imp_ratio)
imp_ratio.to_csv(TABLE_ROOT/'05_사건단위_주요사칭기관_HIGH_MEDIUM.csv', encoding='utf-8-sig')

plot_imp = reliable_imp.groupby(['사기유형','primary_impersonation_subtype']).size().reset_index(name='사건수')
plot_imp['유형내비율'] = plot_imp['사건수'].div(plot_imp.groupby('사기유형')['사건수'].transform('sum')).mul(100)
top_orgs = plot_imp.groupby('primary_impersonation_subtype')['사건수'].sum().nlargest(12).index
plt.figure(figsize=(14,6))
sns.barplot(data=plot_imp[plot_imp.primary_impersonation_subtype.isin(top_orgs)],
            x='primary_impersonation_subtype', y='유형내비율', hue='사기유형')
plt.title('유형별 사건 단위 주요 사칭기관: HIGH·MEDIUM'); plt.xlabel('사칭기관'); plt.ylabel('유형 내 비율(%)')
plt.xticks(rotation=30); plt.tight_layout(); plt.savefig(FIGURE_ROOT/'02_사건단위_사칭기관.png', dpi=170); plt.show()


## 5. 요구 행동·심리전략·금액 EDA


In [ ]:
# 7. 같은 사건의 같은 라벨은 한 번만 계산합니다.
def case_event_ratio(event_df, event_col, output_name):
    if event_df.empty or event_col not in event_df.columns: return pd.DataFrame()
    event = event_df[['case_id',event_col]].dropna().drop_duplicates().merge(type_map, on='case_id', how='inner')
    count = pd.crosstab(event['supervised_target'], event[event_col])
    # 각 유형의 전체 사건 수를 분모로 사용합니다. 여러 행동·전략이 함께 나타나므로 행 합계는 100%를 넘을 수 있습니다.
    total_cases = type_map.groupby('supervised_target')['case_id'].nunique()
    ratio = count.div(total_cases, axis=0).mul(100).round(1)
    count.to_csv(TABLE_ROOT/f'{output_name}_건수.csv', encoding='utf-8-sig')
    ratio.to_csv(TABLE_ROOT/f'{output_name}_비율.csv', encoding='utf-8-sig')
    display(count); display(ratio)
    return ratio

action_ratio = case_event_ratio(tables['actions'], 'action_type', '06_유형별_요구행동')
strategy_ratio = case_event_ratio(tables['strategies'], 'strategy_type', '07_유형별_심리전략')

fig, axes = plt.subplots(2,1,figsize=(15,11))
if not action_ratio.empty:
    sns.heatmap(action_ratio, annot=True, fmt='.1f', cmap='Blues', ax=axes[0])
    axes[0].set_title('사기유형별 요구행동 사건 비율(%)')
if not strategy_ratio.empty:
    sns.heatmap(strategy_ratio, annot=True, fmt='.1f', cmap='Oranges', ax=axes[1])
    axes[1].set_title('사기유형별 심리전략 사건 비율(%)')
plt.tight_layout(); plt.savefig(FIGURE_ROOT/'03_요구행동_심리전략.png', dpi=170); plt.show()


In [ ]:
# 8. 한 사건에서 함께 등장한 심리전략 조합
strategy = tables['strategies'][['case_id','strategy_type']].dropna().drop_duplicates()
strategy_sets = strategy.groupby('case_id')['strategy_type'].apply(lambda x:' + '.join(sorted(set(x)))).reset_index(name='전략조합')
strategy_sets = strategy_sets.merge(type_map, on='case_id', how='left')
combo = strategy_sets.groupby(['supervised_target','전략조합']).size().reset_index(name='사건수')
combo = combo.sort_values(['supervised_target','사건수'], ascending=[True,False])
top_combo = combo.groupby('supervised_target').head(15)
display(top_combo)
top_combo.to_csv(TABLE_ROOT/'08_유형별_심리전략조합_상위.csv', index=False, encoding='utf-8-sig')


In [ ]:
# 9. 피해자에게 요구한 금액과 제시한 금액을 분리
amount = tables['amounts'].copy()
amount['amount_krw'] = pd.to_numeric(amount['amount_krw'], errors='coerce')
amount['amount_10k_krw'] = amount['amount_krw'] / 10000
amount = amount.merge(type_map, on='case_id', how='left')
amount_summary = amount.groupby(['amount_direction','amount_purpose']).agg(
    건수=('amount_event_id','count'), 중앙값_만원=('amount_10k_krw','median'),
    최소_만원=('amount_10k_krw','min'), 최대_만원=('amount_10k_krw','max')
).reset_index().sort_values('건수', ascending=False)
display(amount_summary)
amount_summary.to_csv(TABLE_ROOT/'09_금액방향_용도_요약_만원.csv', index=False, encoding='utf-8-sig')

requested = amount[amount['amount_direction'].eq('REQUESTED_FROM_VICTIM')].copy()
offered = amount[amount['amount_direction'].eq('OFFERED_TO_VICTIM')].copy()
for name, frame in [('피해자에게요구',requested),('피해자에게제시',offered)]:
    frame.to_csv(TABLE_ROOT/f'10_{name}_금액근거.csv', index=False, encoding='utf-8-sig')
print('피해자에게 요구:',len(requested),'건 / 피해자에게 제시:',len(offered),'건')
print('주의: 자동 추출된 통화상 금액이며 실제 피해액이 아닙니다.')


## 6. 텍스트 정리와 학습·검증·테스트 분리


In [ ]:
# 10. 원문은 보존하고 모델 입력용 텍스트만 정리합니다.
def clean_text(value):
    text = unicodedata.normalize('NFKC', str(value or '')).lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'(?m)^\s*(tx|rx|화자\s*\d*|범인|피해자)\s*[:：]\s*', ' ', text)
    text = re.sub(r'\b\d{2,}\b', ' 숫자 ', text)
    return re.sub(r'\s+', ' ', text).strip()

fraud_type = type_df[type_df['supervised_target'].isin(TARGETS)].copy()
fraud_type['clean_text'] = fraud_type['model_input_text'].fillna('').map(clean_text)
fraud_type = fraud_type[fraud_type['clean_text'].str.len() >= MIN_TEXT_LENGTH].copy()
fraud_type['text_hash'] = fraud_type['clean_text'].map(lambda x:hashlib.sha256(x.encode()).hexdigest())
conflict = fraud_type.groupby('text_hash')['supervised_target'].nunique()
fraud_type = fraud_type[~fraud_type['text_hash'].isin(conflict[conflict.gt(1)].index)]
fraud_type = fraud_type.drop_duplicates(['supervised_target','text_hash']).reset_index(drop=True)
assert 'file_id' in fraud_type.columns and fraud_type['file_id'].notna().all()
fraud_type['group_id'] = fraud_type['file_id'].astype(str)
display(fraud_type['supervised_target'].value_counts())


In [ ]:
# 11. 같은 원본 파일의 사건이 서로 다른 세트에 섞이지 않게 분리합니다.
group_label_count = fraud_type.groupby('group_id')['supervised_target'].nunique()
assert group_label_count.max() == 1, '한 원본 파일에 서로 다른 유형 라벨이 있어 그룹 층화가 불가능합니다.'
group_table = fraud_type.groupby('group_id')['supervised_target'].first().reset_index()

train_dev_groups, test_groups = train_test_split(
    group_table, test_size=TEST_RATIO, random_state=SEED, stratify=group_table['supervised_target']
)
train_groups, dev_groups = train_test_split(
    train_dev_groups, test_size=DEV_RATIO, random_state=SEED,
    stratify=train_dev_groups['supervised_target']
)
split_map = {g:'TRAIN' for g in train_groups.group_id}
split_map.update({g:'DEV' for g in dev_groups.group_id})
split_map.update({g:'TEST' for g in test_groups.group_id})
fraud_type['ml_split'] = fraud_type['group_id'].map(split_map)
assert fraud_type.groupby('group_id')['ml_split'].nunique().max() == 1
split_summary = fraud_type.groupby(['ml_split','supervised_target']).size().reset_index(name='사건수')
display(split_summary)
split_summary.to_csv(SPLIT_ROOT/'유형분류_학습검증테스트_분포.csv', index=False, encoding='utf-8-sig')
fraud_type[['case_id','file_id','supervised_target','ml_split']].to_csv(
    SPLIT_ROOT/'유형분류_split_id.csv', index=False, encoding='utf-8-sig'
)


## 7. 대출사기형 vs 수사기관사칭형 지도학습


In [ ]:
# 12. Dummy와 단어·문자 TF-IDF 모델 비교
def candidates():
    return {
        'Dummy':DummyClassifier(strategy='prior'),
        'Word_TFIDF_Logistic':Pipeline([
            ('tfidf',TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=MAX_FEATURES,sublinear_tf=True)),
            ('model',LogisticRegression(max_iter=2000,class_weight='balanced',random_state=SEED))]),
        'Char_TFIDF_Logistic':Pipeline([
            ('tfidf',TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),min_df=2,max_features=MAX_FEATURES,sublinear_tf=True)),
            ('model',LogisticRegression(max_iter=2000,class_weight='balanced',random_state=SEED))]),
        'Char_TFIDF_LinearSVM':Pipeline([
            ('tfidf',TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),min_df=2,max_features=MAX_FEATURES,sublinear_tf=True)),
            ('model',LinearSVC(class_weight='balanced',random_state=SEED))]),
        'Word_TFIDF_ComplementNB':Pipeline([
            ('tfidf',TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=MAX_FEATURES,sublinear_tf=True)),
            ('model',ComplementNB())]),
    }

def multiclass_metrics(y, pred):
    p,r,f1,_ = precision_recall_fscore_support(y,pred,average='macro',zero_division=0)
    return {'accuracy':accuracy_score(y,pred),'macro_precision':p,'macro_recall':r,'macro_f1':f1}

train = fraud_type[fraud_type.ml_split.eq('TRAIN')]
dev = fraud_type[fraud_type.ml_split.eq('DEV')]
test = fraud_type[fraud_type.ml_split.eq('TEST')]
comparison_rows = []
for name, model in candidates().items():
    model.fit(train.clean_text, train.supervised_target)
    pred = model.predict(dev.clean_text)
    comparison_rows.append({'모델':name, **multiclass_metrics(dev.supervised_target,pred)})
    print(name,'완료')
model_comparison = pd.DataFrame(comparison_rows).sort_values(['macro_f1','macro_recall'],ascending=False).reset_index(drop=True)
display(model_comparison)
model_comparison.to_csv(TABLE_ROOT/'11_보이스피싱유형_모델비교_DEV.csv', index=False, encoding='utf-8-sig')


In [ ]:
# 13. 검증 세트로 선정한 모델을 TRAIN+DEV로 재학습하고 TEST는 한 번만 평가합니다.
best_type_name = model_comparison.iloc[0]['모델']
best_type_model = clone(candidates()[best_type_name])
train_dev = fraud_type[fraud_type.ml_split.isin(['TRAIN','DEV'])]
best_type_model.fit(train_dev.clean_text, train_dev.supervised_target)
test_pred = best_type_model.predict(test.clean_text)
test_metrics = multiclass_metrics(test.supervised_target,test_pred)
display(pd.DataFrame([{'모델':best_type_name,**test_metrics}]))
print(classification_report(test.supervised_target,test_pred,zero_division=0))

cm = confusion_matrix(test.supervised_target,test_pred,labels=TARGETS)
plt.figure(figsize=(6,5)); sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',
    xticklabels=[TARGET_KO[x] for x in TARGETS],yticklabels=[TARGET_KO[x] for x in TARGETS])
plt.xlabel('예측'); plt.ylabel('실제'); plt.title('보이스피싱 유형 최종 테스트')
plt.tight_layout(); plt.savefig(FIGURE_ROOT/'04_보이스피싱유형_혼동행렬.png',dpi=170); plt.show()

type_prediction = test[['case_id','file_id','supervised_target','clean_text']].copy()
type_prediction['예측유형'] = test_pred
type_prediction['정답여부'] = type_prediction['supervised_target'].eq(test_pred)
type_prediction = type_prediction.rename(columns={'supervised_target':'실제유형','clean_text':'정제대화문'})
type_prediction.to_csv(PRED_ROOT/'보이스피싱유형_최종테스트_예측_한글.csv',index=False,encoding='utf-8-sig')
joblib.dump(best_type_model, MODEL_ROOT/'fraud_only_type_best_model.joblib')


In [ ]:
# 14. 선정 모델의 유형별 주요 단어 확인
importance_rows = []
if isinstance(best_type_model, Pipeline) and 'tfidf' in best_type_model.named_steps:
    vectorizer = best_type_model.named_steps['tfidf']
    estimator = best_type_model.named_steps['model']
    terms = np.array(vectorizer.get_feature_names_out())
    if hasattr(estimator,'coef_'):
        coef = np.asarray(estimator.coef_)
        values = coef[0] if coef.shape[0] == 1 else coef[list(estimator.classes_).index('LOAN_FRAUD')]
        for idx in values.argsort()[-30:][::-1]:
            importance_rows.append({'유형':'대출사기형 관련','단어_문자조각':terms[idx],'계수':values[idx]})
        for idx in values.argsort()[:30]:
            importance_rows.append({'유형':'수사기관 사칭형 관련','단어_문자조각':terms[idx],'계수':values[idx]})
importance_df = pd.DataFrame(importance_rows)
display(importance_df.head(30))
importance_df.to_csv(TABLE_ROOT/'12_유형분류_주요표현.csv',index=False,encoding='utf-8-sig')
print('주의: 계수는 연관성이지 원인이나 실제 범인의 의도를 확정하는 값이 아닙니다.')


## 8. 행동·심리 특징 기반 보이스피싱 전술 군집


In [ ]:
# 15. 보이스피싱 사건에만 적용할 해석형 특징 규칙
FEATURE_RULES = {
 'money_request':r'송금|이체|입금|납부|지불|결제|돈.{0,8}(보내|내|줘|준비)|금액.{0,8}(보내|입금)',
 'transfer_cash':r'계좌.{0,8}(이체|송금)|현금.{0,8}(인출|찾|전달)|ATM|씨디기|CD기',
 'fee_tax_deposit':r'수수료|선입금|보증금|예치금|공탁금|세금|과태료|벌금|인지대',
 'account_info':r'계좌번호|잔액|통장|카드번호|금융거래|거래내역',
 'personal_info':r'주민번호|주민등록|생년월일|신분증|주소|개인정보|명의',
 'auth_code':r'인증번호|비밀번호|보안카드|OTP|일회용.{0,3}비밀번호',
 'app_remote':r'앱.{0,8}(설치|깔)|어플.{0,8}(설치|깔)|원격.{0,8}(접속|제어)|팀뷰어|퀵서포트',
 'command_pressure':r'하세요|하셔야|해야 합니다|따라 하|지금.{0,8}(가|하|보내|이체)|시키는 대로|말씀드린 대로',
 'urgency_pressure':r'지금 당장|즉시|긴급|오늘 안|시간이 없|빨리|지체하면|마감|몇 분 안',
 'fear_threat':r'체포|구속|압류|범죄|수배|처벌|고소|고발|피해를 입|큰일|위험|납치',
 'isolation_secrecy':r'비밀|말하지 마|알리면 안|누구에게도|혼자만|통화.{0,8}(끊지|유지)|전화.{0,8}(끊지|받지)',
 'authority_trust':r'검찰|검사|경찰|수사관|법원|금융감독원|금감원|은행 본점|정부기관|공문|사건번호',
 'resistance_handling':r'의심|못 믿|확인해 보|그게 아니라|걱정하지|안심|오해|설명드리',
 'benefit_offer':r'대출.{0,10}(승인|가능|해드리)|환급|돌려드리|지원금|혜택|저금리|금리.{0,8}(낮|인하)|한도.{0,8}(상향|증액)',
}
FEATURE_KO = {
 'money_request':'금전·송금요구','transfer_cash':'송금·현금행동','fee_tax_deposit':'수수료·세금·보증금',
 'account_info':'계좌정보','personal_info':'개인정보','auth_code':'인증정보','app_remote':'앱설치·원격접속',
 'command_pressure':'명령·강압','urgency_pressure':'긴급성·시간압박','fear_threat':'공포·위협',
 'isolation_secrecy':'고립·비밀유지','authority_trust':'권위·신뢰형성',
 'resistance_handling':'의심·저항대응','benefit_offer':'이익·혜택제안'
}
compiled_rules = {name:re.compile(pattern,re.I) for name,pattern in FEATURE_RULES.items()}
print('보이스피싱 내부 전술 특징:',len(compiled_rules),'개')


In [ ]:
# 16. 길이 효과를 줄이기 위해 1,000자당 출현 밀도와 존재 여부를 생성합니다.
cluster_source = tables['clustering_ml'].copy()
cluster_source['clean_text'] = cluster_source['model_input_text'].fillna('').map(clean_text)
cluster_source = cluster_source[cluster_source.clean_text.str.len() >= MIN_TEXT_LENGTH].drop_duplicates('case_id').reset_index(drop=True)

def extract_features(text):
    denominator = max(len(text),1)
    row = {}
    present = 0
    for name, pattern in compiled_rules.items():
        count = len(pattern.findall(text))
        row[f'{name}_rate_1k'] = count / denominator * 1000
        row[f'{name}_flag'] = int(count > 0)
        present += int(count > 0)
    row['risk_feature_diversity_ratio'] = present / len(compiled_rules)
    return row

feature_values = pd.DataFrame(cluster_source.clean_text.map(extract_features).tolist())
feature_df = pd.concat([cluster_source[['case_id','file_id','source_category','clean_text']],feature_values],axis=1)
feature_cols = list(feature_values.columns)
feature_df = feature_df.merge(type_map,on='case_id',how='left')
feature_df.to_csv(TABLE_ROOT/'13_보이스피싱_전술특징_데이터.csv',index=False,encoding='utf-8-sig')
display(feature_df.head()); print('군집 입력 특징:',len(feature_cols),'개')


In [ ]:
# 17. K-means 군집 수를 silhouette와 seed 안정성으로 선택합니다.
scaler = StandardScaler()
feature_x = scaler.fit_transform(feature_df[feature_cols])
tactic_rows = []
tactic_models = {}
for k in range(2,8):
    model = KMeans(n_clusters=k,n_init=30,random_state=SEED)
    labels = model.fit_predict(feature_x)
    other_labels = KMeans(n_clusters=k,n_init=30,random_state=SEED+1).fit_predict(feature_x)
    tactic_rows.append({'군집수':k,'silhouette':silhouette_score(feature_x,labels),
                        'seed_stability_ari':adjusted_rand_score(labels,other_labels)})
    tactic_models[k] = (model,labels)
tactic_compare = pd.DataFrame(tactic_rows).sort_values(['silhouette','seed_stability_ari'],ascending=False)
display(tactic_compare)
best_tactic_k = int(tactic_compare.iloc[0]['군집수'])
tactic_model, tactic_labels = tactic_models[best_tactic_k]
feature_df['전술군집ID'] = tactic_labels
tactic_compare.to_csv(TABLE_ROOT/'14_전술군집_K비교.csv',index=False,encoding='utf-8-sig')
feature_df.to_csv(PRED_ROOT/'전술특징_KMeans_군집결과.csv',index=False,encoding='utf-8-sig')
joblib.dump({'scaler':scaler,'model':tactic_model,'feature_columns':feature_cols,'rules':FEATURE_RULES},
            MODEL_ROOT/'fraud_tactic_kmeans.joblib')


In [ ]:
# 18. 군집별 특징 프로필과 기존 사기유형의 관계
tactic_profile = feature_df.groupby('전술군집ID')[feature_cols].mean().T
tactic_profile.to_csv(TABLE_ROOT/'15_전술군집_특징프로필.csv',encoding='utf-8-sig')
display(tactic_profile)
type_alignment = pd.crosstab(feature_df['전술군집ID'],feature_df['supervised_target'].fillna('MIXED_UNKNOWN'),margins=True)
display(type_alignment)
type_alignment.to_csv(TABLE_ROOT/'16_전술군집_기존유형_대응표.csv',encoding='utf-8-sig')

pca = PCA(n_components=2,random_state=SEED)
xy = pca.fit_transform(feature_x)
plot_cluster = pd.DataFrame({'PCA1':xy[:,0],'PCA2':xy[:,1],'전술군집ID':tactic_labels,
                             '기존유형':feature_df['supervised_target'].map(TARGET_KO).fillna('혼합·기타')})
fig,axes=plt.subplots(1,2,figsize=(15,6))
sns.scatterplot(data=plot_cluster,x='PCA1',y='PCA2',hue='전술군집ID',palette='tab10',alpha=.65,ax=axes[0])
axes[0].set_title('행동·심리 특징 K-means 군집')
sns.scatterplot(data=plot_cluster,x='PCA1',y='PCA2',hue='기존유형',alpha=.65,ax=axes[1])
axes[1].set_title('같은 좌표의 기존 사기유형')
plt.tight_layout(); plt.savefig(FIGURE_ROOT/'05_보이스피싱_전술군집.png',dpi=170); plt.show()
print('군집은 정답 분류가 아니며 특징 프로필을 확인한 뒤 사람이 이름을 붙입니다.')


## 9. 텍스트 기반 유사 사건 군집


In [ ]:
# 19. 보이스피싱 텍스트만 TF-IDF → SVD → K-means로 군집화
text_vectorizer = TfidfVectorizer(ngram_range=(1,2),min_df=3,max_features=30000,sublinear_tf=True)
text_x = text_vectorizer.fit_transform(cluster_source.clean_text)
n_components = min(100, text_x.shape[0]-1, text_x.shape[1]-1)
assert n_components >= 2, '텍스트 군집에 필요한 표본 또는 단어가 부족합니다.'
svd = TruncatedSVD(n_components=n_components,random_state=SEED)
text_reduced = svd.fit_transform(text_x)
text_reduced = StandardScaler().fit_transform(text_reduced)

text_rows=[]; text_models={}
for k in range(2,9):
    model=KMeans(n_clusters=k,n_init=30,random_state=SEED)
    labels=model.fit_predict(text_reduced)
    other=KMeans(n_clusters=k,n_init=30,random_state=SEED+1).fit_predict(text_reduced)
    text_rows.append({'군집수':k,'silhouette':silhouette_score(text_reduced,labels),
                      'seed_stability_ari':adjusted_rand_score(labels,other)})
    text_models[k]=(model,labels)
text_k_compare=pd.DataFrame(text_rows).sort_values(['silhouette','seed_stability_ari'],ascending=False)
display(text_k_compare)
best_text_k=int(text_k_compare.iloc[0]['군집수'])
text_model,text_labels=text_models[best_text_k]
text_k_compare.to_csv(TABLE_ROOT/'17_텍스트군집_K비교.csv',index=False,encoding='utf-8-sig')


In [ ]:
# 20. 텍스트 군집별 주요 단어와 대표 사건
text_cluster_result=cluster_source[['case_id','file_id','source_category','clean_text']].copy()
text_cluster_result['텍스트군집ID']=text_labels
terms=np.array(text_vectorizer.get_feature_names_out())
top_term_rows=[]
for cluster_id in range(best_text_k):
    mask=text_labels==cluster_id
    mean_tfidf=np.asarray(text_x[mask].mean(axis=0)).ravel()
    top_terms=terms[mean_tfidf.argsort()[-20:][::-1]]
    top_term_rows.append({'텍스트군집ID':cluster_id,'주요단어_2그램':' | '.join(top_terms)})
top_terms_df=pd.DataFrame(top_term_rows)

distance=text_model.transform(text_reduced)
representative_rows=[]
for cluster_id in range(best_text_k):
    members=np.where(text_labels==cluster_id)[0]
    selected=members[np.argsort(distance[members,cluster_id])[:5]]
    for rank,idx in enumerate(selected,1):
        representative_rows.append({'텍스트군집ID':cluster_id,'대표순위':rank,
            'case_id':text_cluster_result.iloc[idx].case_id,'대표문장':text_cluster_result.iloc[idx].clean_text[:500]})
representative_df=pd.DataFrame(representative_rows)
display(top_terms_df); display(representative_df)
text_cluster_result.to_csv(PRED_ROOT/'텍스트_KMeans_유사사건군집.csv',index=False,encoding='utf-8-sig')
top_terms_df.to_csv(TABLE_ROOT/'18_텍스트군집_주요단어.csv',index=False,encoding='utf-8-sig')
representative_df.to_csv(TABLE_ROOT/'19_텍스트군집_대표사건.csv',index=False,encoding='utf-8-sig')
joblib.dump({'vectorizer':text_vectorizer,'svd':svd,'model':text_model},MODEL_ROOT/'fraud_text_kmeans.joblib')


## 10. 결과 요약과 자동 검증


In [ ]:
# 21. 보이스피싱 단독 분석 보고서와 실행 기록
summary = pd.DataFrame([
    ['보이스피싱 유형 분류',best_type_name,'Macro F1',test_metrics['macro_f1']],
    ['행동·심리 전술 군집',f'K-means k={best_tactic_k}','Silhouette',float(tactic_compare.iloc[0].silhouette)],
    ['텍스트 유사 사건 군집',f'K-means k={best_text_k}','Silhouette',float(text_k_compare.iloc[0].silhouette)],
],columns=['분석','선정방법','평가지표','점수'])
display(summary)
summary.to_csv(REPORT_ROOT/'보이스피싱단독_분석요약.csv',index=False,encoding='utf-8-sig')

report = [
 '# 3단계 V5-2 보이스피싱 단독 분석 결과','', '## Summary','',
 '- 정상 금융상담 데이터를 사용하지 않고 보이스피싱 사례 내부만 분석했습니다.',
 f'- 유형 분류 최종 모델: {best_type_name}',
 f"- 유형 분류 TEST Macro F1: {test_metrics['macro_f1']:.4f}",
 f'- 행동·심리 전술 군집: k={best_tactic_k}',
 f'- 텍스트 유사 사건 군집: k={best_text_k}','',
 '## 해석 기준','',
 '- 지도학습 목표는 원본 폴더의 대출사기형·수사기관사칭형 분류입니다.',
 '- K-means 군집은 정답이 아니라 유사한 전술이나 문구를 가진 탐색적 묶음입니다.',
 '- 사칭·행동·심리·금액은 규칙 기반 SILVER 라벨이므로 탐색적 결과로 해석합니다.',
 '- 금액은 통화 중 자동 추출된 값이며 실제 피해액이 아닙니다.',
 '- 실제 피해 여부 라벨이 없으므로 피해 발생 확률을 예측하지 않았습니다.'
]
(REPORT_ROOT/'03_v5_2_fraud_only_report.md').write_text('\n'.join(report),encoding='utf-8')
manifest = {
 'version':'03_v5_2_fraud_only','dataset_root':str(DATASET_ROOT),'output_root':str(OUTPUT_ROOT),
 'normal_finance_rows_used':0,'seed':SEED,'type_model':best_type_name,'type_test_metrics':test_metrics,
 'tactic_cluster_k':best_tactic_k,'text_cluster_k':best_text_k,
 'group_leakage_check':True,'test_used_once':True,
 'limitations':['SILVER labels','no verified victimization label','FSS corpus only']
}
(REPORT_ROOT/'03_v5_2_run_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2,default=str),encoding='utf-8')

assert fraud_type.groupby('group_id')['ml_split'].nunique().max()==1
assert len(list(MODEL_ROOT.glob('*.joblib'))) >= 3
assert manifest['normal_finance_rows_used'] == 0
print('V5-2 보이스피싱 단독 분석 정상 완료:',OUTPUT_ROOT)
